# Diagnostic: load `gte-multilingual-base` from HuggingFace on serverless (CPU)

Standalone test, separate from `benchmark_gte_french.ipynb`, to get the multilingual GTE
model loading and embedding locally on CPU — with the model **cached in a UC Volume** so it
is downloaded only once.

Two things this notebook establishes:
1. **Volume model cache.** HuggingFace can't download straight into a Volume (its
   symlink/atomic-rename layout breaks on the FUSE mount). So: download to local disk,
   copy into the Volume with symlinks dereferenced (plain files), and on later runs load
   directly from the Volume copy — no re-download. `HF_HOME` stays on local disk for the
   `trust_remote_code` modules cache.
2. **Non-persistent buffers.** The model's custom code registers `position_ids` and the
   RoPE caches (`inv_freq`/`cos_cached`/`sin_cached`) as *non-persistent* buffers, so they
   are absent from the checkpoint. Depending on how transformers/accelerate initialize the
   module they can be left uninitialized → garbage indices → `IndexError` in the RoPE
   branch (`rope_cos[position_ids]`, e.g. "index 45 out of bounds for size 9"). Fix: load
   with `low_cpu_mem_usage=False` and **rebuild those buffers explicitly on CPU** after
   load.

In [0]:
!pip install -q sentence-transformers hf_transfer einops "transformers>=4.41,<5"
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# --- Config + local scratch (writable) ---
import os
import tempfile
import shutil

MODEL = "Alibaba-NLP/gte-multilingual-base"
CATALOG, SCHEMA, VOLUME = "lucasbruand_catalog", "gte_french_bench", "data"
VOLUME_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
VOL_MODEL = f"{VOLUME_ROOT}/models/gte-multilingual-base"   # persistent model cache

spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

# HF_HOME on LOCAL disk: writable scratch for the trust_remote_code modules cache and the
# download's symlink/blob layout (neither works on a Volume).
LOCAL_TMP = tempfile.mkdtemp(prefix="hf_local_")
os.environ["HF_HOME"] = LOCAL_TMP
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"   # fast, robust downloads
print("local HF_HOME:", LOCAL_TMP)
print("volume model cache:", VOL_MODEL)

local HF_HOME: /tmp/hf_local_fx5i5bl4
volume model cache: /Volumes/lucasbruand_catalog/gte_french_bench/data/models/gte-multilingual-base


In [0]:
# --- Get the model into a local dir, using the Volume as the cache ---
from huggingface_hub import snapshot_download

if os.path.isdir(VOL_MODEL) and os.listdir(VOL_MODEL):
    print(f"model cache HIT  -> loading from {VOL_MODEL} (no download)")
    model_path = VOL_MODEL
else:
    print("model cache MISS -> downloading to local disk, then copying to the Volume")
    snap = snapshot_download(MODEL, cache_dir=os.path.join(LOCAL_TMP, "hub"))
    os.makedirs(os.path.dirname(VOL_MODEL), exist_ok=True)
    # symlinks=False dereferences HF's blob symlinks -> self-contained plain files on Volume
    shutil.copytree(snap, VOL_MODEL, symlinks=False, dirs_exist_ok=True)
    print(f"cached model to Volume -> {VOL_MODEL}")
    model_path = VOL_MODEL

print("files:", sorted(os.listdir(model_path))[:12])

/databricks/python_shell/lib/dbruntime/autoreload/discoverability/autoreload_discoverability_hook.py:72: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  return orig_warn(*args, **kwargs)


model cache HIT  -> loading from /Volumes/lucasbruand_catalog/gte_french_bench/data/models/gte-multilingual-base (no download)
files: ['.gitattributes', '1_Pooling', 'README.md', 'config.json', 'images', 'model.safetensors', 'modules.json', 'scripts', 'sentence_bert_config.json', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json']


In [0]:
# --- Load the model (CPU); patch missing mixin methods + rebuild non-persistent buffers ---
import torch
from sentence_transformers import SentenceTransformer
from transformers.modeling_utils import ModuleUtilsMixin

model = SentenceTransformer(
    model_path,
    trust_remote_code=True,
    device="cpu",
    model_kwargs={"low_cpu_mem_usage": False},
)
am = model[0].auto_model

# Some transformers versions don't expose ModuleUtilsMixin helpers on this custom model
# class; its forward() calls get_extended_attention_mask. Bind the ones it needs.
for _m in ("get_extended_attention_mask", "get_head_mask", "invert_attention_mask"):
    if not hasattr(am, _m):
        setattr(type(am), _m, getattr(ModuleUtilsMixin, _m))

# position_ids and the RoPE caches (inv_freq/cos/sin) are NON-persistent buffers, absent from
# the checkpoint; meta/low-cpu-mem init can leave them uninitialized -> garbage indices ->
# IndexError in the RoPE branch. Rebuild them explicitly on CPU.
emb = am.embeddings
mpe = int(am.config.max_position_embeddings)
emb.position_ids = torch.arange(mpe)
re = emb.rotary_emb
re.inv_freq = 1.0 / (re.base ** (torch.arange(0, re.dim, 2).float() / re.dim))
re._set_cos_sin_cache(seq_len=mpe, device=torch.device("cpu"), dtype=torch.float32)

print("position_ids[:8] =", emb.position_ids[:8].tolist())
print("cos_cached shape =", tuple(re.cos_cached.shape))
print("loaded OK; max_seq_length:", model.max_seq_length)

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] NewModel LOAD REPORT from: /Volumes/lucasbruand_catalog/gte_french_bench/data/models/gte-multilingual-base
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


position_ids[:8] = [0, 1, 2, 3, 4, 5, 6, 7]
cos_cached shape = (8192, 64)
loaded OK; max_seq_length: 8192


In [0]:
# --- Embed a few EN/FR sentences and sanity-check ---
import numpy as np

texts = [
    "A girl is styling her hair.",          # en
    "Une fille se coiffe.",                 # fr (same meaning)
    "A man is playing the guitar.",         # en (different)
]
emb = model.encode(texts, normalize_embeddings=True, convert_to_numpy=True)
print("shape:", emb.shape)
print("cos(en-hair, fr-hair)   =", round(float(np.dot(emb[0], emb[1])), 4))
print("cos(en-hair, en-guitar) =", round(float(np.dot(emb[0], emb[2])), 4))
print("\nOK if the cross-lingual same-meaning pair scores clearly higher than the unrelated pair.")

shape: (3, 768)
cos(en-hair, fr-hair)   = 0.8677
cos(en-hair, en-guitar) = 0.3862

OK if the cross-lingual same-meaning pair scores clearly higher than the unrelated pair.
